In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

from students.razin.lesson3 import Exercise

In [ ]:
# Загрузка данных
digits = load_digits()
X = digits.data.astype(np.float32)  # 8x8 пикселей
y = digits.target.astype(np.int64)  #  0-9

# Посмотрим на картинки
print(f"Всего образцов: {X.shape[0]}")
print(f"Пикселей: {X.shape[1]}")
print(f"Классы: {np.unique(y)}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(f"Сифр: {y[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Разделение: train/val/test (60/20/20) с сохранением распределения классов
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)


def normalize(X, min_vals, max_vals):
    # Избегаем деления на ноль
    range_vals = max_vals - min_vals
    range_vals[range_vals == 0] = 1.0
    return 2 * (X - min_vals) / range_vals - 1


X_train = normalize(X_train, train_min, train_max).astype(np.float32)
X_val = normalize(X_val, train_min, train_max).astype(np.float32)
X_test = normalize(X_test, train_min, train_max).astype(np.float32)

# Проверка результата
print("\nПосле нормализации:")
print(f"Train диапазон: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Val диапазон:   [{X_val.min():.3f}, {X_val.max():.3f}]")
print(f"Test диапазон:  [{X_test.min():.3f}, {X_test.max():.3f}]")
print(f"\nРазмеры выборок: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

In [ ]:
def evaluate_loss(model, loss_fn, X, y):
    output = model.forward(X)
    return loss_fn.forward(output, y)


def evaluate_accuracy(model, X, y):
    logits = model.forward(X)
    predictions = np.argmax(logits, axis=1)
    return np.mean(predictions == y)


def train_epoch(model, loss_fn, X, y, lr, batch_size):
    n_samples = X.shape[0]
    indices = np.random.permutation(n_samples)
    X_shuffled = X[indices]
    y_shuffled = y[indices]

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        x_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        # Forward pass
        output = model.forward(x_batch)
        loss_value = loss_fn.forward(output, y_batch)

        # Backward pass
        grad = loss_fn.backward()
        model.backward(grad)

        for param, grad_param in zip(model.parameters, model.grad, strict=True):
            param -= lr * grad_param

    return loss_value


def train_and_track(model, loss_fn, X_tr, y_tr, X_vl, y_vl, lr, epochs, batch_size, verbose=False, verbose_every=10):
    history = {"train_losses": [], "val_losses": [], "val_accs": [], "epochs": []}

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, loss_fn, X_tr, y_tr, lr, batch_size)

        if epoch % verbose_every == 0 or epoch == 1:
            vl_loss = evaluate_loss(model, loss_fn, X_vl, y_vl)
            vl_acc = evaluate_accuracy(model, X_vl, y_vl)

            history["train_losses"].append(train_loss)
            history["val_losses"].append(vl_loss)
            history["val_accs"].append(vl_acc)
            history["epochs"].append(epoch)

            if verbose:
                print(f"Epoch {epoch:3d}: Train Loss={train_loss:.4f} | Val Loss={vl_loss:.4f} | Val Acc={vl_acc:.3f}")

    return history

In [ ]:
print("=" * 70)
print("перебор всех гиперпараметров")
print("=" * 70)

# Определяем диапазоны для поиска
lr_candidates = [0.001, 0.005, 0.1, 0.5, 1.0, 5.0, 10.0]
hidden_candidates = [1, 2, 4, 8, 16, 32, 64, 128]
batch_candidates = [1, 2, 4, 8, 16, 32, 64, 128]
N_EPOCHS = 25

# Хранилище результатов
all_results = {}
best_val_acc = 0
best_params = None
best_model = None
best_history = None

total_combinations = len(lr_candidates) * len(hidden_candidates) * len(batch_candidates)
print(f"\nВсего комбинаций для проверки: {total_combinations}")
print(f"LR варианты: {lr_candidates}")
print(f"Hidden варианты: {hidden_candidates}")
print(f"Batch варианты: {batch_candidates}")
print(f"Эпох на комбинацию: {N_EPOCHS}\n")

# Перебираем все комбинации
for idx, (lr, hidden, batch) in enumerate(itertools.product(lr_candidates, hidden_candidates, batch_candidates), 1):
    print(f"\n[{idx}/{total_combinations}] Проверка: lr={lr}, hidden={hidden}, batch={batch}")
    print("-" * 50)

    # Фиксируем random seed для воспроизводимости
    rng = np.random.default_rng(42)

    # Создаем модель
    model = Exercise.create_model(
        Exercise.create_linear_layer(64, hidden, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss_fn = Exercise.create_cross_entropy_loss()

    # Обучаем (verbose=False чтобы не засорять вывод)
    history = train_and_track(
        model,
        loss_fn,
        X_train,
        y_train,
        X_val,
        y_val,
        lr=lr,
        epochs=N_EPOCHS,
        batch_size=batch,
        verbose=False,
        verbose_every=10,
    )

    # Берем последние значения
    final_val_acc = history["val_accs"][-1]
    final_val_loss = history["val_losses"][-1]
    final_train_loss = history["train_losses"][-1]

    # Сохраняем результат
    all_results[(lr, hidden, batch)] = {
        "val_acc": final_val_acc,
        "val_loss": final_val_loss,
        "train_loss": final_train_loss,
        "history": history,
    }

    # Обновляем лучший результат
    if final_val_acc > best_val_acc:
        best_val_acc = final_val_acc
        best_params = {"lr": lr, "hidden": hidden, "batch": batch}
        best_model = model
        best_history = history
        print(f"🎉 НОВЫЙ ЛУЧШИЙ РЕЗУЛЬТАТ! Val Acc={final_val_acc:.4f}, Val Loss={final_val_loss:.4f}")
    else:
        print(f"Результат: Val Loss={final_val_loss:.4f}, Val Acc={final_val_acc:.4f}")

assert best_params is not None, "не установлены лучшие параметры"

# Выводим лучшие параметры
print("\n" + "=" * 70)
print("ЛУЧШИЕ НАЙДЕННЫЕ ПАРАМЕТРЫ")
print("=" * 70)
print(f"Learning Rate: {best_params['lr']}")
print(f"Hidden Size: {best_params['hidden']}")
print(f"Batch Size: {best_params['batch']}")
print(f"Validation Accuracy: {best_val_acc:.4f}")
print(
    f"Validation Loss: {all_results[(best_params['lr'], best_params['hidden'], best_params['batch'])]['val_loss']:.4f}"
)

In [ ]:
print("=" * 70)
print("ФИНАЛЬНОЕ ОБУЧЕНИЕ с лучшими параметрами")
print("=" * 70)

# Создаем новую модель с лучшими параметрами
rng = np.random.default_rng(42)
final_model = Exercise.create_model(
    Exercise.create_linear_layer(64, best_params["hidden"], rng),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(best_params["hidden"], 10, rng),
    Exercise.create_logsoftmax_layer(),
)
final_loss = Exercise.create_cross_entropy_loss()

# Обучаем с увеличенным числом эпох и выводом прогресса
history_final = train_and_track(
    final_model,
    final_loss,
    X_train,
    y_train,
    X_val,
    y_val,
    lr=best_params["lr"],
    epochs=25,
    batch_size=best_params["batch"],
    verbose=True,
    verbose_every=5,
)

# ========== ИТОГОВЫЕ МЕТРИКИ ==========
print("\n" + "=" * 70)
print("ИТОГОВЫЕ МЕТРИКИ")
print("=" * 70)
print(f"Гиперпараметры: lr={best_params['lr']}, hidden={best_params['hidden']}, batch={best_params['batch']}\n")

print(f"{'Выборка':<8} {'Loss':>12} {'Accuracy':>12}")
print("-" * 34)

for name, X_set, y_set in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    loss_val = evaluate_loss(final_model, final_loss, X_set, y_set)
    acc_val = evaluate_accuracy(final_model, X_set, y_set)
    print(f"{name:<8} {loss_val:>12.4f} {acc_val:>12.3f}")

test_acc = evaluate_accuracy(final_model, X_test, y_test)
print(f"\nОсновной результат: Test Accuracy = {test_acc:.3f}")

y_pred = np.argmax(final_model.forward(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix на тестовой выборке", fontsize=14)
plt.tight_layout()
plt.show()

# График обучения финальной модели
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_final["epochs"], history_final["train_losses"], label="Train Loss", marker="o")
plt.plot(history_final["epochs"], history_final["val_losses"], label="Val Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss during training")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_final["epochs"], history_final["val_accs"], label="Val Accuracy", marker="o", color="green")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy during training")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()